# SecureMemo AI Agentic Workflow

In [1]:
# LLM Initialization

In [2]:
import os

from google.colab import userdata

# Configure the API key as an environment variable

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

print("✓ API key configured successfully!")

✓ API key configured successfully!


### Imports and Installations

In [3]:
# PyPDF and Text Splitting/Chunking Libraries
!pip install -qU langchain-community pypdf langchain-text-splitters rank-bm25
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
# Google LLM
!pip install -qU langchain-google-genai
# Vector Database
!pip install -qU --upgrade chromadb

In [5]:
# Embeddings and Storage
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [6]:
# Generation
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [7]:
# Tool Decorator
from langchain_core.tools import tool

In [8]:
# Create Agent
from langchain.agents import create_agent

In [9]:
# LangChain Installs
!pip install langsmith langchain

In [10]:
# LangChain Debugging

import os

from google.colab import userdata

os.environ["LANGSMITH_TRACING"] = "true"

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

os.environ["LANGSMITH_PROJECT"] = "SecureMemo_AI_SubAgent_Tests"

#Keyword Search Function

In [11]:
from rank_bm25 import BM25Okapi
import re

def keyword_search(query, chunks, top_k=3):
    """
    Performs BM25 keyword search over a list of text chunks.

    Args:
        query (str): The user's search query.
        chunks (list): List of strings (the text chunks).
        top_k (int): Number of results to return.

    Returns:
        list: The top_k most relevant chunks.
    """
    # Simple Tokenization/Preprocessing (Low-level cleaning)
    def tokenize(text):
        return re.sub(r'[^\w\s]', '', text.lower()).split()

    tokenized_corpus = [tokenize(chunk) for chunk in chunks]
    tokenized_query = tokenize(query)

    # Initialize BM25
    bm25 = BM25Okapi(tokenized_corpus)

    # Retrieve top chunks
    top_chunks = bm25.get_top_n(tokenized_query, chunks, n=top_k)

    return top_chunks

## RAG Tool For Project Description

### Company Project Notes - Document Loading and Chunking

In [12]:
# Project Description Notes
loader_pd1 = PyPDFLoader("Project_Descriptions1.pdf")
document_pd1 = loader_pd1.load()

# Combine all pages into one text string
full_text_pd1 = "\n\n".join([page.page_content for page in document_pd1])
print(f"Total text length: {len(full_text_pd1)} characters")
#text = document_pd1[1].page_content

Total text length: 4468 characters


In [13]:
text_splitter_pd1 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, separators=["\n\n", "\n", ". ", " ", ""])
chunks_pd1 = text_splitter_pd1.split_text(full_text_pd1)

In [14]:
print(f"Number of chunks: {len(chunks_pd1)}")
print()
for i, chunk in enumerate(chunks_pd1):
    print("Chunk", i, "has",len(chunk), "Characters:")
    print(chunk)
    print()

Number of chunks: 6

Chunk 0 has 950 Characters:
N ‘nd R Associates  
Financial Industry - Ongoing Project Descriptions 
1) Small Business Loan Support Program 
Description: 
 
 The Small Business Loan Support Program helps small and growing businesses secure 
financing to expand operations, purchase equipment, and manage daily expenses. The team 
reviews applications, analyzes financial statements, assesses creditworthiness, and structures 
loan packages, while also tracking repayments and providing guidance on budgeting and cash 
flow management. Using loan management software, spreadsheet analysis tools, and customer 
relationship management systems, employees streamline application processing, maintain 
accurate records, and monitor client interactions. By coordinating these efforts, small and 
medium-sized enterprises will be able to access funding efficiently, reduce the risk of loan 
defaults, and build a foundation for long-term growth. 
Employee Involvement and Access Levels:


### Embeddings and Storage

In [15]:
# Using the model "models/gemini-embedding-001"
# If there is an error with this cell, restart the colab runtime session and run all cells again in order
embeddings_pd1 = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# Use Chroma.from_texts() with chunks and embedding model
vectorstore_pd1 = Chroma.from_texts(
    texts=chunks_pd1,
    embedding=embeddings_pd1,
    collection_name="my_documents_collection"
)

print(f"Stored {vectorstore_pd1._collection.count()} chunks in the vector store")

Stored 6 chunks in the vector store


In [16]:
# Preview what is stored in the vector database
sample_pd1 = vectorstore_pd1._collection.peek(limit=1)
print(f"Sample chunk ID: {sample_pd1['ids']}")
print(f"Sample text: {sample_pd1['documents']}")
print(f"Embedding length: {len(sample_pd1['embeddings'][0])}")

Sample chunk ID: ['feb67236-4708-4ab7-a452-488e1de067ad']
Sample text: ['N ‘nd R Associates  \nFinancial Industry - Ongoing Project Descriptions \n1) Small Business Loan Support Program \nDescription: \n \n The Small Business Loan Support Program helps small and growing businesses secure \nfinancing to expand operations, purchase equipment, and manage daily expenses. The team \nreviews applications, analyzes financial statements, assesses creditworthiness, and structures \nloan packages, while also tracking repayments and providing guidance on budgeting and cash \nflow management. Using loan management software, spreadsheet analysis tools, and customer \nrelationship management systems, employees streamline application processing, maintain \naccurate records, and monitor client interactions. By coordinating these efforts, small and \nmedium-sized enterprises will be able to access funding efficiently, reduce the risk of loan \ndefaults, and build a foundation for long-term growth. \nEm

### Retrieval

In [17]:
query1 = "What are the titles of the ongoing projects in this company?"
results = vectorstore_pd1.similarity_search(query1, k=5)

# Print the results
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content)
    print("---")

Result 1:
N ‘nd R Associates  
Financial Industry - Ongoing Project Descriptions 
1) Small Business Loan Support Program 
Description: 
 
 The Small Business Loan Support Program helps small and growing businesses secure 
financing to expand operations, purchase equipment, and manage daily expenses. The team 
reviews applications, analyzes financial statements, assesses creditworthiness, and structures 
loan packages, while also tracking repayments and providing guidance on budgeting and cash 
flow management. Using loan management software, spreadsheet analysis tools, and customer 
relationship management systems, employees streamline application processing, maintain 
accurate records, and monitor client interactions. By coordinating these efforts, small and 
medium-sized enterprises will be able to access funding efficiently, reduce the risk of loan 
defaults, and build a foundation for long-term growth. 
Employee Involvement and Access Levels:
---
Result 2:
2) Digital Loan Processin

### Generation

In [18]:
llm_pd1 = ChatGoogleGenerativeAI(model="gemini-flash-latest")

template_pd1 = """You are a helpful project description notes agent. Answer the question in a concise and straightforward way. Provide more details about the projects when asked for them. Only use information from this context. If query is out of context, respond that you cannot provide the response."

Context:
{context}

Question: {question}

Answer:"""

prompt_pd1 = ChatPromptTemplate.from_template(template_pd1)

In [19]:
@tool
def process_project_description(query: str) -> str:
  """Search and retrieve specific information about company projects,
    including project titles, summaries, Employee Involvement and Access Levels."""
  # Running semantic search
  vector_docs = vectorstore_pd1.similarity_search(query, k=5)
  vector_texts = [doc.page_content for doc in vector_docs]

  # Running Keyword search
  keyword_texts = keyword_search(query, chunks_pd1, top_k=3)

  # Merging duplicate chunks
  merged_texts = list(set(vector_texts + keyword_texts))
  context_string = "\n\n---\n\n".join(merged_texts)
  chain = prompt_pd1 | llm_pd1 | StrOutputParser()
  return chain.invoke({"context": context_string, "question": query})

In [20]:
# Test with the provided queries
test_queries_pd1 = [
    "What are the titles of the ongoing projects?",
    "Can you give a summary of each project?",
    "Which project requires the most amount of senior employees with high access levels?",
    "What interns do these projects need?",
    "Can you give me the company's sensitive financial information?"
]

for query in test_queries_pd1:
    print(f"Q: {query}")
    response = process_project_description.invoke(query)
    print(f"A: {response}")
    print("=" * 60)

Q: What are the titles of the ongoing projects?
A: The titles of the ongoing projects are:
1. Small Business Loan Support Program
2. Digital Loan Processing and Risk Assessment Initiative
3. Personal Investment and Financial Planning Services Program
Q: Can you give a summary of each project?
A: Based on the context provided, here are the summaries for each project:

*   **Small Business Loan Support Program:** This program assists small businesses in securing financing for operations and equipment. The team analyzes financial statements, structures loan packages, and provides guidance on cash flow management to help businesses grow and reduce default risks.
*   **Digital Loan Processing and Risk Assessment Initiative:** This initiative focuses on automating loan applications and approvals. It uses risk assessment analytics and workflow automation to verify documents, detect fraud, and increase the speed and accuracy of the lending process.
*   **Personal Investment and Financial Plann

In [21]:
from langchain.agents import create_agent

agent_llm_pd1 = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

system_prompt_pd1 = """You are a helpful project description notes assistant.
Your goal is to help users understand company projects using the 'get_project_details' tool.
Always verify project facts using the tool before answering.
If a user asks for sensitive financial info which user is not authorized to access or things not in the documents,
politely decline as per company policy."""

agent_pd1 = create_agent(
    model=agent_llm_pd1,
    tools=[process_project_description],
    system_prompt=system_prompt_pd1
)

# confirmation print command
# print("Project Description Agent with One RAG Tool is ready")

In [22]:
prompts = [
    "What are the names of the ongoing projects mentioned in the notes?",
    "What does a Finance Intern do in the Small Business Loan Support Program?",
    "I am Finance Intern, which project requires the most senior-level employees with high or admin access?",
    "Are there any projects that do not require an intern?",
    "What is the company's annual revenue for 2025 and who is the CEO?",
    "Can you give me the private bank account numbers for the clients in the Personal Investment program?"
]

for query in prompts:
    response = agent_pd1.invoke({"messages": [{"role": "human", "content": query}]})
    print(f"=== RESPONSE for: '{query}' ===")
    print(response["messages"][-1].content)
    print("\n")

=== RESPONSE for: 'What are the names of the ongoing projects mentioned in the notes?' ===
Okay, I found the names of the ongoing projects. They are: 1) Small Business Loan Support Program, 2) Digital Loan Processing and Risk Assessment Initiative, and 3) Personal Investment and Financial Planning Services Program.


=== RESPONSE for: 'What does a Finance Intern do in the Small Business Loan Support Program?' ===
Okay, the Finance Intern in the Small Business Loan Support Program is responsible for data entry, preparing client documents, and updating basic records. They also have low access within the program.


=== RESPONSE for: 'I am Finance Intern, which project requires the most senior-level employees with high or admin access?' ===
The Digital Loan Processing and Risk Assessment Initiative requires the most senior-level employees with high or admin access, totaling three roles: Information Technology Executive / System Administrator, Senior Credit Risk Analyst, and Project Manager

## RAG Tool for Meeting Notes

### Meeting Notes - Document Loading and Chunking

In [23]:
# PyPDF and Text Splitting/Chunking Libraries
# !pip install -qU langchain-community pypdf langchain-text-splitters
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

In [24]:
# First Meeting Notes to be stored in meeting notes storage
loader_mn1 = PyPDFLoader("Meeting_Notes1.pdf")
document_mn1 = loader_mn1.load()

# Combine all pages into one text string
full_text_mn1 = "\n\n".join([page.page_content for page in document_mn1])
# print(f"Total text length: {len(full_text_mn1)} characters")
#text = document_mn1[1].page_content

In [25]:
text_splitter_mn1 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])

In [26]:
chunks_mn1 = text_splitter_mn1.split_text(full_text_mn1)

In [27]:
print(f"Number of chunks: {len(chunks_mn1)}")
print()
for i, chunk in enumerate(chunks_mn1):
    print("Chunk", i, "has",len(chunk), "Characters:")
    print(chunk)
    print()

Number of chunks: 11

Chunk 0 has 184 Characters:
Meeting Notes - N 'nd R Associates 
 
● Manager mentioned that maintaining accurate client communications and up-to-date 
reporting is important to ensure overall project coordination.

Chunk 1 has 140 Characters:
● Review loan applications and financial statements to identify areas that may need 
further analysis (Small Business Loan Support Program).

Chunk 2 has 196 Characters:
● Opportunities for process improvements and technology updates were explored, with 
some items flagged for closer review. 
 
● Conduct market research and track investment trends to inform future

Chunk 3 has 191 Characters:
recommendations (Personal Investment and Financial Planning Services Program). 
 
● Updates to client profiles and monitoring progress in portfolio and CRM systems need to 
be done regularly.

Chunk 4 has 123 Characters:
be done regularly. 
 
● Monitor credit risk and repayment trends to catch emerging patterns and review 
potential conce

In [28]:
# Second Meeting Notes
loader_mn2 = PyPDFLoader("Meeting_Notes2.pdf")
document_mn2 = loader_mn2.load()

# Combine all pages into one text string
full_text_mn2 = "\n\n".join([page.page_content for page in document_mn2])
# print(f"Total text length: {len(full_text_mn2)} characters")
#text = document_mn2[1].page_content

In [29]:
text_splitter_mn2 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])
chunks_mn2 = text_splitter_mn2.split_text(full_text_mn2)

In [30]:
print(f"Number of chunks: {len(chunks_mn2)}")
print()
for i, chunk in enumerate(chunks_mn2):
    print("Chunk", i, "has",len(chunk), "Characters:")
    print(chunk)
    print()

Number of chunks: 9

Chunk 0 has 162 Characters:
Meeting Notes – N 'nd R Associates 
● Review client loan histories and repayment performance using loan management 
software (Small Business Loan Support Program)

Chunk 1 has 134 Characters:
software (Small Business Loan Support Program) 
 
● Validate submitted documents and cross-check compliance requirements with workflow

Chunk 2 has 185 Characters:
automation and document management tools (Digital Loan Processing and Risk 
Assessment Initiative) 
 
● Maintain client data and investment records in CRM and portfolio tracking systems

Chunk 3 has 174 Characters:
● Analyze trends in outstanding loans and assess risk patterns using analytics platforms 
 
● Conduct financial market scans and monitor investment performance with financial

Chunk 4 has 178 Characters:
analytics tools (Personal Investment and Financial Planning Services Program) 
 
● Update and reconcile client portfolios, including investment allocations and projected 
retur

### Embeddings and Storage

In [31]:
# Using the model "models/gemini-embedding-001"
# If there is an error with this cell, restart the colab runtime session and run all cells again in order
embeddings_mn = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

meeting_notes_chunks = chunks_mn1 + chunks_mn2

# Use Chroma.from_texts() with chunks and embedding model
vectorstore_mn = Chroma.from_texts(
    texts=chunks_mn1,
    embedding=embeddings_mn,
    collection_name="meeting_notes_collection"
)

vectorstore_mn = Chroma.from_texts(
    texts=chunks_mn2,
    embedding=embeddings_mn,
    collection_name="meeting_notes_collection"
)

print(f"Stored {vectorstore_mn._collection.count()} chunks in the vector store")

Stored 20 chunks in the vector store


In [32]:
#Preview what is stored in the vector database
sample_mn = vectorstore_mn._collection.peek(limit=4)
print(f"Sample chunk ID: {sample_mn['ids']}")
print(f"Sample text: {sample_mn['documents']}")
print(f"Embedding length: {len(sample_mn['embeddings'][0])}")

Sample chunk ID: ['71176d1e-e8db-49d6-9235-b16a9df92798', 'faeef85c-94c5-4e82-9369-7043aed23519', 'e0598b49-e9e4-4f14-aaac-ea838c094ff5', 'bd3753de-e9d3-4582-9103-a6b60f8fd428']
Sample text: ["Meeting Notes - N 'nd R Associates \n \n● Manager mentioned that maintaining accurate client communications and up-to-date \nreporting is important to ensure overall project coordination.", '● Review loan applications and financial statements to identify areas that may need \nfurther analysis (Small Business Loan Support Program).', '● Opportunities for process improvements and technology updates were explored, with \nsome items flagged for closer review. \n \n● Conduct market research and track investment trends to inform future', 'recommendations (Personal Investment and Financial Planning Services Program). \n \n● Updates to client profiles and monitoring progress in portfolio and CRM systems need to \nbe done regularly.']
Embedding length: 3072


### Retrieval

In [33]:
# Test retrieval using similarity_search method
query1 = "What tasks are related to the Small Business Loan Support Program?"
results = vectorstore_mn.similarity_search(query1, k=5)

# Print the results
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content)
    print("___")

Result 1:
software (Small Business Loan Support Program) 
 
● Validate submitted documents and cross-check compliance requirements with workflow
___
Result 2:
● Review loan applications and financial statements to identify areas that may need 
further analysis (Small Business Loan Support Program).
___
Result 3:
Meeting Notes – N 'nd R Associates 
● Review client loan histories and repayment performance using loan management 
software (Small Business Loan Support Program)
___
Result 4:
automation and document management tools (Digital Loan Processing and Risk 
Assessment Initiative) 
 
● Maintain client data and investment records in CRM and portfolio tracking systems
___
Result 5:
● Basic updates are needed, such as entering client information, maintaining records, and 
monitoring routine metrics
___


### Generation

In [34]:
# Initialize the LLM (matching the exact format from Project Description)
llm_mn = ChatGoogleGenerativeAI(model="gemini-flash-latest")

template_mn = """You are a helpful assistant analyzing meeting notes from a company.

Use the following context from meeting notes to answer the question. If you cannot find the answer in the context, say so.

Context:
{context}

Question: {question}

Answer:"""

prompt_mn = ChatPromptTemplate.from_template(template_mn)

In [35]:
@tool
def process_meeting_notes(query: str) -> str:
  """Process and Search the uploaded Meeting Notes for tasks mentioned in company meetings. Tasks can include information about action items for projects, types of employees and access levels required, and deadines.
    Use this tool when asked about tasks mentioned in meeting notes."""
  # Running semantic search
  vector_docs = vectorstore_mn.similarity_search(query, k=5)
  vector_texts = [doc.page_content for doc in vector_docs]

  # Running Keyword search
  keyword_texts = keyword_search(query, meeting_notes_chunks, top_k=3)

  # Merging duplicate chunks
  merged_texts = list(set(vector_texts + keyword_texts))
  context_string = "\n\n---\n\n".join(merged_texts)
  chain = prompt_mn | llm_mn | StrOutputParser()
  return chain.invoke({"context": context_string, "question": query})

### Testing Meeting Notes RAG

In [36]:
# Test with the provided queries
test_queries_mn = [
    "What tasks are related to the Small Business Loan Support Program?",
    "What compliance activities were mentioned in the meetings?",
    "What client-related updates need to be done?",
    "What tasks can be completed by interns?",
    "Which tasks need immediate action and have deadlines approaching soon?"
]

for query in test_queries_mn:
    print(f"Q: {query}")
    response = process_meeting_notes.invoke(query)
    print(f"A: {response}")
    print("=" * 60)

Q: What tasks are related to the Small Business Loan Support Program?
A: Based on the meeting notes, the tasks related to the **Small Business Loan Support Program** are:

*   Validating submitted documents and cross-checking compliance requirements with workflow.
*   Reviewing loan applications and financial statements to identify areas that may need further analysis.
*   Reviewing client loan histories and repayment performance using loan management software.
Q: What compliance activities were mentioned in the meetings?
A: Based on the meeting notes, the compliance activities mentioned include:

*   **Verification and validation of client documents:** Ensuring submitted documents are accurate and complete.
*   **Compliance checks:** General reviews to ensure standards are met.
*   **Cross-checking compliance requirements with workflow:** Specifically mentioned in relation to the Small Business Loan Support Program software.
Q: What client-related updates need to be done?
A: Based on 

## Meeting Notes Agent with RAG Tool

In [37]:
from langchain.agents import create_agent

agent_llm_mn = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

system_prompt_mn = """You are a helpful meeting notes assistant for processing and analyzing company tasks in the provided meeting notes.

You have one tool:
1. process_meeting_notes: Use this tool for any questions about the tasks in the meeting notes.
These questions can include, but are not limited to, knowing the tasks for a specific project, finding
types of tasks that require a certain type of employee position and/or employee access level, or knowing the deadlines for tasks.

Only use this tool for answering questions about what is in the provided meeting notes."""

agent_mn = create_agent(
    model=agent_llm_mn,
    tools=[process_meeting_notes],
    system_prompt=system_prompt_mn
)

# print("Meeting Notes Agent with One RAG Tool is ready.")

### Testing for Meeting Notes Agent

In [38]:
agent_query1_mn = agent_mn.invoke({
    "messages": [{"role": "human", "content": """What tasks can be completed by interns?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query1_mn["messages"][-1].content)

=== AGENT RESPONSE ===
I cannot find the answer in the context. There is no mention of "interns" or tasks specifically assigned to them in the provided notes.


In [39]:
agent_query2_mn = agent_mn.invoke({
    "messages": [{"role": "human", "content": """Which tasks are client-facing?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query2_mn["messages"][-1].content)

=== AGENT RESPONSE ===
The client-facing tasks include:

*   **Scheduling client interactions:** Explicitly mentioned as a task to be tracked.
*   **Providing recommendations and services:** Part of the "Personal Investment and Financial Planning Services Program."
*   **Generating client reports and dashboards:** Creating materials through reporting platforms and CRM for client use.


In [40]:
agent_query3_mn = agent_mn.invoke({
    "messages": [{"role": "human", "content": """Which are some technology platforms mentioned in the tasks?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query3_mn["messages"][-1].content)

=== AGENT RESPONSE ===
The technology platforms mentioned in the tasks are: Small Business Loan Support Program, Digital Loan Processing and Risk Assessment Initiative, CRM, Portfolio tracking systems, Reporting platforms, Analytics platforms, and Automation tools.


In [41]:
agent_query4_mn = agent_mn.invoke({
    "messages": [{"role": "human", "content": """What tasks seem easy to finish quicky that an intern could do?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query4_mn["messages"][-1].content)

=== AGENT RESPONSE ===
I cannot find the answer in the context. The provided meeting notes do not mention "interns." However, the notes do identify "simpler tasks" and "basic updates," which include:

*   Verification of client documents and compliance checks
*   Entering client information
*   Maintaining records and documentation across systems
*   Monitoring routine metrics and logging metrics
*   Recording routine updates


## RAG Tool for Employee Data

### Employee Records - Document Loading and Chunking

In [42]:
# Install pandas and openpyxl for Excel file handling
!pip install -qU pandas openpyxl
import pandas as pd

# Load the Excel file
employee_df = pd.read_excel("Sample_Employee_Data.xlsx")

print(f"Loaded {len(employee_df)} employees")
print(f"\nColumns: {list(employee_df.columns)}")
print(f"\nFirst few employees:")
print(employee_df.head())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
Loaded 50 employees

Columns: ['Name', 'Email', 'Employee ID', 'Team', 'Position']

First few employees:
              Name                          Email Employee ID         Team  \
0     James 

In [43]:
# Convert the DataFrame to text format for RAG
# Each employee becomes a document chunk
employee_texts = []
for _, row in employee_df.iterrows():
    employee_text = f"""Employee: {row['Name']}
Email: {row['Email']}
Employee ID: {row['Employee ID']}
Team: {row['Team']}
Position: {row['Position']}"""
    employee_texts.append(employee_text)

print(f"Created {len(employee_texts)} employee documents")
print(f"\nSample employee document:")
print(employee_texts[0])

Created 50 employee documents

Sample employee document:
Employee: James Carter
Email: james.carter@securememo.ai
Employee ID: EMP001
Team: Strategy
Position: Project Manager


In [44]:
# Create additional contextual chunks about teams and positions
# This helps with queries like "who are the engineers?" or "who are the managers?"
team_summary = employee_df.groupby('Team').size().to_dict()
position_summary = employee_df.groupby('Position').size().to_dict()

team_text = "Team Summary:\n" + "\n".join([f"- {team}: {count} employees" for team, count in team_summary.items()])
position_text = "Position Summary:\n" + "\n".join([f"- {position}: {count} employees" for position, count in position_summary.items()])

# Add summary texts to help with aggregate queries
employee_texts.append(team_text)
employee_texts.append(position_text)

print(f"\nTotal chunks including summaries: {len(employee_texts)}")
print(f"\nTeam summary:")
print(team_text)


Total chunks including summaries: 52

Team summary:
Team Summary:
- Engineering: 13 employees
- Finance: 8 employees
- HR: 7 employees
- Marketing: 10 employees
- Strategy: 12 employees


### Embeddings and Storage

In [45]:
# Create embeddings and vector store for employee data
# Using the model "models/gemini-embedding-001"
embeddings_ed = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# Create the vector store
vectorstore_ed = Chroma.from_texts(
    texts=employee_texts,
    embedding=embeddings_ed,
    collection_name="employee_data_collection"
)

print(f"Stored {vectorstore_ed._collection.count()} chunks in the employee vector store")

Stored 52 chunks in the employee vector store


In [46]:
# Preview what's stored in the vector database
sample_ed = vectorstore_ed._collection.peek(limit=3)
print(f"Sample employee chunk IDs: {sample_ed['ids'][:3]}")
print(f"\nSample text preview:")
print(sample_ed['documents'][0])
print(f"\nEmbedding length: {len(sample_ed['embeddings'][0])}")

Sample employee chunk IDs: ['bd49730b-7f1a-481d-bb50-b83a2584f000', '4ed363f7-a9e6-4c34-b34e-a3b25796bc72', 'b5e612b9-3625-4036-9149-22ebbf5d640c']

Sample text preview:
Employee: James Carter
Email: james.carter@securememo.ai
Employee ID: EMP001
Team: Strategy
Position: Project Manager

Embedding length: 3072


### Retrieval

In [47]:
# Test retrieval using similarity_search method
query_ed = "Who are the project managers?"
results_ed = vectorstore_ed.similarity_search(query_ed, k=5)

# Print the results
for i, doc in enumerate(results_ed):
    print(f"Result {i+1}:")
    print(doc.page_content)
    print("___")

Result 1:
Employee: Lucas Mendes
Email: lucas.mendes@securememo.ai
Employee ID: EMP036
Team: Strategy
Position: Project Manager
___
Result 2:
Employee: Ethan Scott
Email: ethan.scott@securememo.ai
Employee ID: EMP034
Team: Engineering
Position: Project Manager
___
Result 3:
Position Summary:
- Accounts Officer: 2 employees
- Administrative Intern: 2 employees
- Client Service Executive: 3 employees
- Client Support Assistant: 3 employees
- Compliance Officer: 3 employees
- Credit Analyst: 2 employees
- Data Entry Intern: 2 employees
- Finance Intern: 3 employees
- Finance Manager: 3 employees
- Financial Advisor: 3 employees
- IT Executive: 3 employees
- Junior Business Development Executive: 1 employees
- Loan Administrator: 1 employees
- Loan Support Assistant: 1 employees
- Operations Officer: 2 employees
- Portfolio Manager: 1 employees
- Project Manager: 3 employees
- Risk Analyst: 1 employees
- Senior Credit Risk Analyst: 2 employees
- Senior Investment Analyst: 3 employees
- Sen

### Generation

In [48]:
# Initialize LLM for employee queries
llm_ed = ChatGoogleGenerativeAI(model="gemini-flash-latest")

template_ed = """You are a helpful HR assistant with access to company employee records.

Use the following employee information to answer the question. Be concise and accurate.
If the question asks about multiple people (e.g., "who are the managers?"), list all relevant employees.
If you cannot find the information, say so clearly.

Employee Information:
{context}

Question: {question}

Answer:"""

prompt_ed = ChatPromptTemplate.from_template(template_ed)

In [49]:
@tool
def process_employee_data(query: str) -> str:
  """Process and Search the uploaded Employee Data for information about employees, teams, positions, and clearance levels.
    Use this tool when asked about employees, their roles, team composition, or access levels."""
  # Running semantic search
  vector_docs = vectorstore_ed.similarity_search(query, k=5)
  vector_texts = [doc.page_content for doc in vector_docs]

  # Running Keyword search
  keyword_texts = keyword_search(query, employee_texts, top_k=3)

  # Merging duplicate chunks
  merged_texts = list(set(vector_texts + keyword_texts))
  context_string = "\n\n---\n\n".join(merged_texts)
  chain = prompt_ed | llm_ed | StrOutputParser()
  return chain.invoke({"context": context_string, "question": query})

In [50]:
# Test the employee data RAG chain
test_queries_ed = [
    "Who are the project managers?",
    "Which employees work in the Engineering team?",
    "What is Priya Nair's position and email?",
    "Who are the interns?",
    "How many people are on the Finance team?",
    "Who is EMP014?"
]

for query in test_queries_ed:
    print(f"Q: {query}")
    response = process_employee_data.invoke(query)
    print(f"A: {response}")
    print("=" * 60)

Q: Who are the project managers?
A: The project managers are James Carter, Ethan Scott, and Lucas Mendes.
Q: Which employees work in the Engineering team?
A: The employees identified in the Engineering team are:

*   Tobias Müller
*   Nadia Okonkwo
*   Dominic Russo
*   Ethan Scott
*   Jonas Berg
Q: What is Priya Nair's position and email?
A: Priya Nair's position is System Administrator and her email is priya.nair@securememo.ai.
Q: Who are the interns?
A: The interns are:
* Camille Dubois (Administrative Intern)
* Jasmine Lee (Data Entry Intern)
* Aaliyah Grant (Finance Intern)
* Chloe Martin (Administrative Intern)
Q: How many people are on the Finance team?
A: There are 8 employees on the Finance team.
Q: Who is EMP014?
A: Tyler Brooks (Senior Credit Risk Analyst, Finance)


## Employee Data Agent with RAG Tool


In [51]:
from langchain.agents import create_agent

agent_llm_ed = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

system_prompt_ed = """You are a helpful HR assistant for N 'nd R Associates with access to employee records.

You have one tool:
1. process_employee_data: Use this tool for any questions about employees, teams, positions, or organizational structure.
These questions can include, but are not limited to, finding specific employees by name or ID, finding employees by team or position,
determining who has certain clearance levels, or understanding the organizational structure.

Only use this tool for answering questions about what is in the employee database."""

agent_ed = create_agent(
    model=agent_llm_ed,
    tools=[process_employee_data],
    system_prompt=system_prompt_ed
)

print("Employee Data Agent with One RAG Tool is ready.")

Employee Data Agent with One RAG Tool is ready.


## Testing for Employee Data Agent


In [52]:
# Test 1: Find employees by role
agent_query1_ed = agent_ed.invoke({
    "messages": [{"role": "human", "content": """Who are the project managers?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query1_ed["messages"][-1].content)

=== AGENT RESPONSE ===
The project managers are:
- James Carter
- Ethan Scott
- Lucas Mendes


In [53]:
# Test 2: Find employees by team
agent_query2_ed = agent_ed.invoke({
    "messages": [{"role": "human", "content": """Which employees work in the Engineering team?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query2_ed["messages"][-1].content)

=== AGENT RESPONSE ===
The employees identified in the Engineering team are:

*   Tobias Müller
*   Nadia Okonkwo
*   Dominic Russo
*   Ethan Scott
*   Jonas Berg


In [54]:
# Test 3: Find specific employee information
agent_query3_ed = agent_ed.invoke({
    "messages": [{"role": "human", "content": """What is Priya Nair's position and email?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query3_ed["messages"][-1].content)

=== AGENT RESPONSE ===
Priya Nair is a System Administrator and her email is priya.nair@securememo.ai.


In [55]:
# Test 4: Find employees by position type
agent_query4_ed = agent_ed.invoke({
    "messages": [{"role": "human", "content": """Who are the interns?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query4_ed["messages"][-1].content)

=== AGENT RESPONSE ===
The interns are:
- Aisha Rahman (Data Entry Intern)
- Jasmine Lee (Data Entry Intern)
- Isaac Fernandez (Finance Intern)
- Aaliyah Grant (Finance Intern)


In [56]:
# Test 5: Team composition query
agent_query5_ed = agent_ed.invoke({
    "messages": [{"role": "human", "content": """How many people work on the Finance team?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query5_ed["messages"][-1].content)

=== AGENT RESPONSE ===
There are 8 employees on the Finance team.


In [57]:
# Test 6: Find employee by ID
agent_query6_ed = agent_ed.invoke({
    "messages": [{"role": "human", "content": """Who is employee EMP014?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query6_ed["messages"][-1].content)

=== AGENT RESPONSE ===
Employee EMP014 is Tyler Brooks, a Senior Credit Risk Analyst on the Finance team.


In [58]:
# Test 7: Find employees suitable for specific tasks
agent_query7_ed = agent_ed.invoke({
    "messages": [{"role": "human", "content": """Who would be qualified to handle senior-level financial analysis tasks?"""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query7_ed["messages"][-1].content)

=== AGENT RESPONSE ===
Fatou Diallo, Tyler Brooks, and Patrick O'Connor are qualified to handle senior-level financial analysis tasks.


##Main Orchestration Agent

### Create Agent with the three RAG Agents as Sub-Agents

This approach references: https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant

- Create Subagents (Done in above code cells)
- Create tool functions to define when the main agent should call/invoke the other agents
- Use the main agent to answer a query like "Please assign employees to the tasks outlined in the meeting notes. Reference our project description document for finding the best matches.".

In [59]:
# Tool for project description agent
@tool
def project_description_agent(request:str) -> str:
  """ Call the agent for processing the project description.

  Use this agent when the main agent wants to gather information about the projects in the company, the details of each project,
  and the employee positions involved in each project.

  Input: Natural language project information gathering request (e.g., 'Get the project title, description, and employee positions for each project.')

  """
  result = agent_pd1.invoke({
        "messages": [{"role": "user", "content": request}]
    })

  return result["messages"][-1].content

In [60]:
# Tool for meeting notes agent
@tool
def meeting_notes_agent(request:str) -> str:
  """ Call the agent for processing the meeting notes.

  Use this agent when the main agent wants to get the tasks in the meeting notes.
  Input: Natural language tasks request (e.g., 'Get the tasks mentioned in the meeting notes.')

  """
  result = agent_mn.invoke({
        "messages": [{"role": "user", "content": request}]
    })

  return result["messages"][-1].content

In [61]:
# Tool for employee data agent
@tool
def employee_data_agent(request:str) -> str:
  """ Call the agent for processing the employee data.

  Use this agent when the main agent wants to find an employee for a task based on employee position.
  Input: Natural language employee request (e.g., 'Find the name and email of an employee that is a Finance Intern with low-level access.')

  """
  result = agent_ed.invoke({
        "messages": [{"role": "user", "content": request}]
    })

  return result["messages"][-1].content

In [62]:
# Create the main agent
agent_llm_main = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

system_prompt_agent_main = """You are a helpful main assistant to N nd' R Associates with access to all three subagents:
the project description agent, the meeting notes agent, and the employee data agent.

You are responsible for pairing employees to company tasks outlined in the meeting notes. The employee-task assignments will be the final output you give to the user.

You have three subagents:
1. project_description_agent: Use this tool first to invoke the project description agent.
Get context about the projects in the company such as the project titles, descriptions of projects, and the employee positions associated with each project.

2. meeting_notes_agent: Use this tool second to invoke the meeting notes agent.
Get the tasks from the meeting notes and understand what employee positions the tasks are looking for.
Use the information from the project description agent to connect tasks to projects if applicable.

3. employee_data_agent: Use this tool third to invoke the employee data agent.
Get the employees in this data that are suitable (have the right position and access level) for the tasks returned from the meeting notes agent.

Only use these three subagents for assigning employees to tasks outlined in the meeting notes. Use the project description agent to get guidance/context about the tasks.

The final output should be the assignments of company employees to tasks from the meeting notes. If a task cannot be matched, say so.
"""

agent_main = create_agent(
    model=agent_llm_main,
    tools=[project_description_agent, meeting_notes_agent, employee_data_agent],
    system_prompt=system_prompt_agent_main
)

print("Main Agent with Access to Three Sub Agents is Ready.")

Main Agent with Access to Three Sub Agents is Ready.


In [63]:
# Give the initial query to the main agent
initial_query = """Please assign employees to the tasks outlined in the meeting notes.
Reference our project description document for finding the best matches."""

response_main = agent_main.invoke({
    "messages": [{"role": "human", "content": initial_query}]
})

print("=== MAIN AGENT RESPONSE ===")
print(response_main["messages"][-1].content)

=== MAIN AGENT RESPONSE ===
Here are the employee assignments to tasks:

**Compliance and Documentation:**
*   Camille Dubois (Administrative Intern)

**Financial Management:**
*   Isaac Fernandez (Finance Intern)
*   Tanya Petrova (Finance Intern)
*   Aaliyah Grant (Finance Intern)

**Research and Analysis:**
*   Oliver Bennett (Junior Business Development Executive)

The remaining tasks cannot be matched to an employee with the available data. These include:
*   System and Process Operations
*   Reporting
*   Client and Decision Management
*   Monitoring


In [64]:
# Test 2: Ask about a specific project's staffing
agent_query2_main = agent_main.invoke({
    "messages": [{"role": "human", "content": """Using the meeting notes and employee data,
which employees are best suited for tasks in the Small Business Loan Support Program?
Reference the project description for context on what positions are needed."""}]
})

print("=== AGENT RESPONSE ===")
print(agent_query2_main["messages"][-1].content)

=== AGENT RESPONSE ===
Okay, I have the project details, tasks, and employee data. Here are the employee assignments:

**Small Business Loan Support Program Task Assignments:**

*   **Validate documents:**
    *   Isaac Fernandez (Finance Intern, Strategy Team)
    *   Tanya Petrova (Finance Intern, Finance Team)
    *   Aaliyah Grant (Finance Intern, Strategy Team)
    *   Note: Access levels for validating documents cannot be confirmed.
*   **Analyze financials:**
    *   Liam O'Brien (Credit Analyst, liam.obrien@securememo.ai)
    *   Omar Khalil (Credit Analyst, omar.khalil@securememo.ai)
    *   Note: Access levels are not specified in the employee data.
*   **Review loan history:**
    *   Ryan Castillo (Loan Administrator)
    *   Note: Access levels are not specified in the employee data.


In [65]:
# Test 3: Check intern assignments
agent_query3_main = agent_main.invoke({
    "messages": [{"role": "human", "content": "Which tasks from the meeting notes can be handled by interns, and who are the available interns?"}]
})

print("=== AGENT RESPONSE ===")
print(agent_query3_main["messages"][-1].content)

=== AGENT RESPONSE ===
Okay, I have gathered the following information:

**Tasks from Meeting Notes:** Documentation and Compliance, Financial Adjustments, Research and Analysis, Systems and Operations, Reporting, Client Management, and Monitoring.

**Interns and Projects:**

*   **Small Business Loan Support Program:** Finance Intern (Low access)
*   **Digital Loan Processing and Risk Assessment Initiative:** Data Entry Intern / Loan Support Assistant (Low access)
*   **Personal Investment and Financial Planning Services Program:** Administrative Intern / Client Support Assistant (Low access)

**Available Interns:**

*   **Finance Interns:** Isaac Fernandez (isaac.fernandez@securememo.ai), Tanya Petrova (tanya.petrova@securememo.ai), Aaliyah Grant (aaliyah.grant@securememo.ai)
*   **Data Entry Interns:** Aisha Rahman (aisha.rahman@securememo.ai), Jasmine Lee (jasmine.lee@securememo.ai)
*   **Administrative Interns:** Camille Dubois (camille.dubois@securememo.ai), Chloe Martin (chloe.m

In [66]:
# Test 4: Unassignable task check
agent_query4_main = agent_main.invoke({
    "messages": [{"role": "human", "content": "Are there any tasks in the meeting notes that cannot be matched to any current employee?"}]
})

print("=== AGENT RESPONSE ===")
print(agent_query4_main["messages"][-1].content)

=== AGENT RESPONSE ===
Based on the meeting notes, the tasks are: Administrative & Documentation, Client & Financial Management, Reporting & Analysis, Operational & Technical, and Research & Monitoring.

Based on the employee data, the employee positions are: Compliance Officer, Project Manager, IT Executive, System Administrator.

Based on the project descriptions, the employee positions are: Finance Intern, Junior Business Development Executive, Senior Loan Officer, Credit Analyst, Loan Administrator, Accounts Officer, Finance Manager, Data Entry Intern / Loan Support Assistant, Information Technology Executive / System Administrator, Operations Officer, Senior Credit Risk Analyst, Compliance Officer, Project Manager, Administrative Intern / Client Support Assistant, Financial Advisor, Senior Investment Analyst, Portfolio Manager, Risk Analyst, Client Service Executive.

Matching tasks to employee positions:
*   Administrative & Documentation: Matches Administrative Intern / Client S